# 💾 Lesson 09 · Checkpoint 持久化

> **会话可恢复**：`InMemorySaver` + `thread_id` 保存图状态，支持多步恢复与中断续跑。

本课你会学到：
- 为什么没有 Checkpoint 就很难做可靠多轮
- `thread_id` 如何标识一条会话线程
- 如何读取 / 续跑已保存的 checkpoint

模型：**DeepSeek V4 Pro**（`deepseek-v4-pro`）。`.env` 需配置 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_BASE_URL`）。


## 逻辑总览

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "13px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#94a3b8"
  }
}}%%
flowchart LR
    S([START]) --> J[generate_joke]
    J --> X[generate_explanation]
    X --> E([END])

    CP[(InMemorySaver<br/>thread_id)] -.-> J
    CP -.-> X


    classDef input fill:#BFDBFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
    classDef llm fill:#FED7AA,stroke:#F97316,color:#9A3412,stroke-width:2px
    classDef branch fill:#E9D5FF,stroke:#A855F7,color:#6B21A8,stroke-width:2px
    classDef tool fill:#BBF7D0,stroke:#22C55E,color:#14532D,stroke-width:2px
    classDef output fill:#FECACA,stroke:#F87171,color:#7F1D1D,stroke-width:2px
    class S input
    class J,X llm
    class CP tool
    class E output
```

**要点：** 没有 Checkpoint 就没有可靠多轮会话；`thread_id` 是会话钥匙。



# Imports

注意引入 `InMemorySaver`（内存版 Checkpointer）。


In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver


C:\Users\86137\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\triton\windows_utils.py:372: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


In [2]:
load_dotenv()

True

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

llm = ChatOpenAI(
    model="deepseek-v4-pro",
    temperature=0,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    timeout=120,
    max_tokens=1200,
    max_retries=1,
    extra_body={"thinking": {"type": "disabled"}},
)


# 定义 State


In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

# Node — generate_joke


In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

# Node — generate_explanation


In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

# 编译图并挂上 Checkpointer

## 逻辑总览

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "13px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#94a3b8"
  }
}}%%
flowchart LR
    S([START]) --> J[generate_joke]
    J --> X[generate_explanation]
    X --> E([END])

    CP[(InMemorySaver<br/>thread_id)] -.-> J
    CP -.-> X


    classDef input fill:#BFDBFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
    classDef llm fill:#FED7AA,stroke:#F97316,color:#9A3412,stroke-width:2px
    classDef branch fill:#E9D5FF,stroke:#A855F7,color:#6B21A8,stroke-width:2px
    classDef tool fill:#BBF7D0,stroke:#22C55E,color:#14532D,stroke-width:2px
    classDef output fill:#FECACA,stroke:#F87171,color:#7F1D1D,stroke-width:2px
    class S input
    class J,X llm
    class CP tool
    class E output
```

**要点：** 没有 Checkpoint 就没有可靠多轮会话；`thread_id` 是会话钥匙。

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

# 用 thread_id 执行

同一 `thread_id` = 同一会话；换 id 就是新线程。


In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'足球'}, config=config1)
workflow.invoke({'topic':'大白菜'}, config=config1)

{'topic': '大白菜',
 'joke': 'Why did the Chinese cabbage get invited to every party?\n\nBecause it was the ultimate 大白菜 (dà bái cài) — it always brought the best *wrapped* gifts and never caused any *beef*! 🥬',
 'explanation': 'This joke is a clever multilingual pun that blends English wordplay with Mandarin Chinese. Here’s the breakdown:\n\n**1. The Homophone Hook (大白菜)**\nThe core of the joke lies in the Chinese term **大白菜 (dà bái cài)** , which literally means "Chinese cabbage" (or Napa cabbage).\nHowever, when spoken aloud, "dà bái cài" sounds almost identical to the English phrase **"The Best Wrapped"** (with a playful accent/pronunciation shift: "da bai cai" → "the best wrapped"). The joke sets up the cabbage as the "ultimate" version of itself, but the punchline reveals it\'s actually the "ultimate" party guest because of this pun.\n\n**2. The Double Meaning of "Wrapped"**\nThe joke says it brought the best **"wrapped" gifts**. This works on two levels:\n- **Literal:** Cabbage lea

# 查看 Checkpoint

读取当前线程快照。


In [9]:
import json


def show_state(snap, title=None):
    """把 StateSnapshot 打成可读 JSON。"""
    if title:
        print(f"=== {title} ===")
    payload = {
        "values": dict(snap.values),
        "next": list(snap.next),
        "config": snap.config,
        "metadata": snap.metadata,
        "created_at": snap.created_at,
    }
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))


show_state(workflow.get_state(config1), "thread_id=1 当前快照")

=== thread_id=1 当前快照 ===
{
  "values": {
    "topic": "大白菜",
    "joke": "Why did the Chinese cabbage get invited to every party?\n\nBecause it was the ultimate 大白菜 (dà bái cài) — it always brought the best *wrapped* gifts and never caused any *beef*! 🥬",
    "explanation": "This joke is a clever multilingual pun that blends English wordplay with Mandarin Chinese. Here’s the breakdown:\n\n**1. The Homophone Hook (大白菜)**\nThe core of the joke lies in the Chinese term **大白菜 (dà bái cài)** , which literally means \"Chinese cabbage\" (or Napa cabbage).\nHowever, when spoken aloud, \"dà bái cài\" sounds almost identical to the English phrase **\"The Best Wrapped\"** (with a playful accent/pronunciation shift: \"da bai cai\" → \"the best wrapped\"). The joke sets up the cabbage as the \"ultimate\" version of itself, but the punchline reveals it's actually the \"ultimate\" party guest because of this pun.\n\n**2. The Double Meaning of \"Wrapped\"**\nThe joke says it brought the best **\"wrapp

# 查看历史 Checkpoint

`get_state_history` 列出该线程的状态时间线。


In [10]:
for i, snap in enumerate(workflow.get_state_history(config1)):
    show_state(snap, f"history[{i}] · step={snap.metadata.get('step')} · next={list(snap.next)}")
    print()

=== history[0] · step=6 · next=[] ===
{
  "values": {
    "topic": "大白菜",
    "joke": "Why did the Chinese cabbage get invited to every party?\n\nBecause it was the ultimate 大白菜 (dà bái cài) — it always brought the best *wrapped* gifts and never caused any *beef*! 🥬",
    "explanation": "This joke is a clever multilingual pun that blends English wordplay with Mandarin Chinese. Here’s the breakdown:\n\n**1. The Homophone Hook (大白菜)**\nThe core of the joke lies in the Chinese term **大白菜 (dà bái cài)** , which literally means \"Chinese cabbage\" (or Napa cabbage).\nHowever, when spoken aloud, \"dà bái cài\" sounds almost identical to the English phrase **\"The Best Wrapped\"** (with a playful accent/pronunciation shift: \"da bai cai\" → \"the best wrapped\"). The joke sets up the cabbage as the \"ultimate\" version of itself, but the punchline reveals it's actually the \"ultimate\" party guest because of this pun.\n\n**2. The Double Meaning of \"Wrapped\"**\nThe joke says it brought the b

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'美少女'}, config=config2)

{'topic': '美少女',
 'joke': '为什么美少女拍照时总爱比“耶”？\n\n因为她们想证明——  \n“美”是真的，“少”是假的！ ✌️😄',
 'explanation': '这个笑话利用了中文里“美少女”这个词的双关含义，玩了一个文字游戏。\n\n**拆解一下：**\n\n1.  **“美少女”的字面意思：**\n    -   “美” = 美丽\n    -   “少” = 年少、年轻\n    -   “女” = 女性\n    -   合起来就是“美丽年轻的女性”。\n\n2.  **拍照比“耶”✌️的视觉含义：**\n    -   这个手势在拍照时非常常见，通常代表“耶”（胜利/开心）或者单纯为了造型。\n    -   **关键点**：这个手势在视觉上，就是一个数字“2”。\n\n3.  **笑话的包袱（笑点）：**\n    -   当美少女比出“耶✌️”时，她仿佛在用手势对“美少女”这个词进行“事实核查”。\n    -   **“美”是真的**：她竖起的两根手指，可以看作是在确认第一个字“美”——我很美，这是真的（打勾✅）。\n    -   **“少”是假的**：同时，这个“2”的手势，又像是在拆穿第二个字“少”——年龄上可能已经不再“年少”了，所以“少”这个字是假的（打叉❌，或者说是个“二”）。\n\n所以，这个笑话的幽默感来自于，用一个天真可爱的拍照手势，巧妙地自我调侃了“美少女”这个称呼中“年轻”这一要素的真实性。😄'}

In [12]:
show_state(workflow.get_state(config2), "thread_id=2 当前快照")

=== thread_id=2 当前快照 ===
{
  "values": {
    "topic": "美少女",
    "joke": "为什么美少女拍照时总爱比“耶”？\n\n因为她们想证明——  \n“美”是真的，“少”是假的！ ✌️😄",
    "explanation": "这个笑话利用了中文里“美少女”这个词的双关含义，玩了一个文字游戏。\n\n**拆解一下：**\n\n1.  **“美少女”的字面意思：**\n    -   “美” = 美丽\n    -   “少” = 年少、年轻\n    -   “女” = 女性\n    -   合起来就是“美丽年轻的女性”。\n\n2.  **拍照比“耶”✌️的视觉含义：**\n    -   这个手势在拍照时非常常见，通常代表“耶”（胜利/开心）或者单纯为了造型。\n    -   **关键点**：这个手势在视觉上，就是一个数字“2”。\n\n3.  **笑话的包袱（笑点）：**\n    -   当美少女比出“耶✌️”时，她仿佛在用手势对“美少女”这个词进行“事实核查”。\n    -   **“美”是真的**：她竖起的两根手指，可以看作是在确认第一个字“美”——我很美，这是真的（打勾✅）。\n    -   **“少”是假的**：同时，这个“2”的手势，又像是在拆穿第二个字“少”——年龄上可能已经不再“年少”了，所以“少”这个字是假的（打叉❌，或者说是个“二”）。\n\n所以，这个笑话的幽默感来自于，用一个天真可爱的拍照手势，巧妙地自我调侃了“美少女”这个称呼中“年轻”这一要素的真实性。😄"
  },
  "next": [],
  "config": {
    "configurable": {
      "thread_id": "2",
      "checkpoint_ns": "",
      "checkpoint_id": "1f19212f-21ba-6b6d-8002-945e78c1fd25"
    }
  },
  "metadata": {
    "source": "loop",
    "step": 2,
    "parents": {}
  },
  "created_at": "2026-08-07T03:49:09.416228+00:00"


In [13]:
for i, snap in enumerate(workflow.get_state_history(config2)):
    show_state(snap, f"history[{i}] · step={snap.metadata.get('step')} · next={list(snap.next)}")
    print()

=== history[0] · step=2 · next=[] ===
{
  "values": {
    "topic": "美少女",
    "joke": "为什么美少女拍照时总爱比“耶”？\n\n因为她们想证明——  \n“美”是真的，“少”是假的！ ✌️😄",
    "explanation": "这个笑话利用了中文里“美少女”这个词的双关含义，玩了一个文字游戏。\n\n**拆解一下：**\n\n1.  **“美少女”的字面意思：**\n    -   “美” = 美丽\n    -   “少” = 年少、年轻\n    -   “女” = 女性\n    -   合起来就是“美丽年轻的女性”。\n\n2.  **拍照比“耶”✌️的视觉含义：**\n    -   这个手势在拍照时非常常见，通常代表“耶”（胜利/开心）或者单纯为了造型。\n    -   **关键点**：这个手势在视觉上，就是一个数字“2”。\n\n3.  **笑话的包袱（笑点）：**\n    -   当美少女比出“耶✌️”时，她仿佛在用手势对“美少女”这个词进行“事实核查”。\n    -   **“美”是真的**：她竖起的两根手指，可以看作是在确认第一个字“美”——我很美，这是真的（打勾✅）。\n    -   **“少”是假的**：同时，这个“2”的手势，又像是在拆穿第二个字“少”——年龄上可能已经不再“年少”了，所以“少”这个字是假的（打叉❌，或者说是个“二”）。\n\n所以，这个笑话的幽默感来自于，用一个天真可爱的拍照手势，巧妙地自我调侃了“美少女”这个称呼中“年轻”这一要素的真实性。😄"
  },
  "next": [],
  "config": {
    "configurable": {
      "thread_id": "2",
      "checkpoint_ns": "",
      "checkpoint_id": "1f19212f-21ba-6b6d-8002-945e78c1fd25"
    }
  },
  "metadata": {
    "source": "loop",
    "step": 2,
    "parents": {}
  },
  "created_at": "2026-08-07T03:49:09.4